In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import os
import sys
import time
import copy
import numpy as np
import scipy
import jax
import jax.numpy as jnp
import viser

sys.path.append(os.path.abspath(".."))
from environments import ClothEnv
from planners import gpu_sls

In [ ]:
server = viser.ViserServer()
_ = server.scene.add_grid(name="ground", position=(0,0,-0.009))

In [ ]:
params = {
    'time_step': 0.02,       # s
    'thickness': 1e-3,       # m
    'youngs_modulus': 1e5,   # Pa
    'possion_ratio': 0.3,
    'mass_density': 100,     # kg/m³
    'num_floating_grippers': 1,
    'grip_stiffness': 5000,
    'contact_smoothing': 1e-4,
}
env = ClothEnv.from_obj_file("assets/shirt.obj", **params)

In [ ]:
def make_control_constraints(
    u_min: jnp.ndarray,
    u_max: jnp.ndarray,
):
    def constraints(x, u, t):
        u = u.reshape(-1, 4)
        control_constraints = jnp.concatenate((u - u_max, u_min - u)).ravel()
        return control_constraints

    return constraints


def make_constant_disturbance(alpha: float):
    def disturbance(X: jnp.ndarray) -> jnp.ndarray:
        N, nx = X.shape
        E0 = alpha * jnp.eye(nx, dtype=X.dtype)
        return jnp.broadcast_to(E0, (N, nx, nx))

    return disturbance


# Initial state
x_grip = jnp.array([[0, 0.2, 0.5e-3]])
state0 = env.state(x_grip=x_grip)

# Target state
R = scipy.spatial.transform.Rotation.from_rotvec([0, 0, np.deg2rad(90)])
R = jnp.array(R.as_matrix())
x_node = env.params.x_node_rest @ R
x_node = env.params.x_node_rest.at[:, 1].add(0.33)
state_goal = env.state(x_node=x_node)

# Initial control (zero velocity)
control0 = env.control(c_grip=jnp.array([1.0]))


N = 10  # Planning horizon
dt = env.params.dt

def cost(W, reference, x, u, t):
    x_ref = state_goal
    u_ref = control0
    state_err = x - x_ref
    control_err = u - u_ref
    return (
        1.0 * jnp.sum(state_err[:-3]**2)
        + 0.5 * jnp.sum(control_err[:-1]**2)
    )

def dynamics(x, u, t, parameter):
    return env.step(x, u)


vmax = 0.3
u_max = jnp.array([vmax, vmax, vmax, 10.0])
constraints = make_control_constraints(u_min=-u_max, u_max=u_max)

admm_cfg = gpu_sls.ADMMConfig(
    eps_abs=5e-2,
    eps_rel=1e-2,
    rho_max=1e3,
    max_iterations=100,
    rho_update_frequency=25,
    initial_rho=10.0,
)

sls_cfg = gpu_sls.SLSConfig(
    max_sls_iterations=2,
    sls_primal_tol=1e-2,
    enable_fastsls=False,
    initialize_nominal=True,
    max_initial_sqp_iterations=0,
    warm_start=False,
    rti=False,
)

sqp_cfg = gpu_sls.SQPConfig(
    max_sqp_iterations=1,
    warm_start=False,
    feas_tol=1e-2,
    step_tol=1e-4,
    line_search=True,
)

cfg = gpu_sls.MPCConfig(
    n=state0.size,
    nu=control0.size,
    N=N,
    dt=dt,
    W=None,
    u_ref=control0,
)

obstacles = jnp.zeros((0, 3))
disturbance = make_constant_disturbance(alpha=0.003 * dt)
nc = constraints(state0, control0, 0.0).size

mpc_controller = gpu_sls.GenericMPC(
    sls_cfg,
    sqp_cfg,
    admm_cfg,
    config=cfg,
    dynamics=dynamics,
    cost=cost,
    constraints=constraints,
    obstacles=obstacles,
    disturbance=disturbance,
    num_constraints=nc,
    shift=1,
    X_in=None,
    U_in=None,
)

In [ ]:
X_in = jnp.tile(state0[None, :], (N + 1, 1))
U_in = jnp.tile(control0[None, :], (N, 1))
# U_in = U_in.at[:16, 2].set(-0.2)
# for i in range(U_in.shape[0]):
#     X_in = X_in.at[i+1].set(env.step(X_in[i], U_in[i]))

controller = copy.deepcopy(mpc_controller)
controller.U0 = U_in
controller.X0 = X_in

state = state0
env.visualize(server, state)

for i in range(100):
    start = time.time()
    u0, X_pred, U_pred, V_pred, backoffs, Phi_x, Phi_u = controller.run(x0=state, reference=None, parameter=None)
    elapsed = time.time() - start
    print(f"MPC step took {elapsed * 1e3:.2f} ms")

    print("u0 =", u0)
    control = jnp.clip(u0 if not jnp.isnan(u0).any() else control0, -u_max, u_max)
    state = env.step(state, control)

    if jnp.isnan(state).any():
        raise RuntimeError("NaN occurred in rope state")
    env.visualize(server, state)

    wait = dt - elapsed
    if wait > 0:
        time.sleep(wait)